# Safari Compass Calibration

The Safari-Zone analog of the **Metronome Compass Calibration** notebook.  It
identifies the loaded seed, plans the manual advances needed to encounter a
Metang, identifies the *battle* seed from the safari encounter (bait / mud /
ball), checks how confident that identification is, saves the run, and feeds the
shared timer→frame calibration model.

## Two seeds, two kinds of "frame" (read this first)

Both seeds are fixed by **game-frame (clock) timing** — the timer precision we
calibrate:

- **Seed A** — the overworld stream: encounters, roamer relocation, Elm calls.
- **Seed B** — the battle stream: hits, crits, capture / flee odds.

An **advance frame** ("advance") is how many times a seed's state has been
advanced via `advance_rng`, driven by **player actions, not the clock**.  Section A
walks *Seed A's* advance frame (Elm calls + chatot flips + Sweet Scent) purely so
that we encounter a Metang — this does **not** affect Seed B or the calibration.
Calibration is the same timer(M)→Seed-B-frame fit as metronome; safari just
identifies Seed B differently and may carry a slightly different load-screen
offset, applied as a separate **safari offset** (β/slope stays from metronome).

## Sections
- **A** — identify Seed A (roamer + Elm), then plan the advances to a Metang.
- **B** — identify Seed B via safari compass, then a confidence / neighbor check.
- **C** — save the run to `data/safari_runs.jsonl`.
- **D** — analysis over the saved runs.
- **E** — apply the safari offset to `data/calibration_model.json` (offset only).

In [60]:
%load_ext autoreload
%autoreload 2
import datetime as dt

from utils.calibration_tools import (
    # Section A -- roamer routes + Elm seed identification (shared with metronome)
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section C -- persist a safari run
    save_safari_run,
    # Section D -- analysis
    load_safari_runs,
    fit_safari_offset,
    # Section E -- apply the safari offset (deliberate; review-then-confirm)
    update_safari_offset,
)
from utils.safari_advance import (
    advance_context, context_from_row,
    identify_frame, prompt_target_frame,
    plan_advances, margin_guide, describe_plan,
)
from utils.safari_confidence import path_confidence, print_confidence

from claytonlib.compass import compass_safari, CompassSafariInput
from claytonlib.calibration import CalibrationModel
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.chart import STRATEGY_ONLY_BALLS, CRITERIA_CAPTURE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Section A.1 — Roamer + Elm identification  (→ `a_seed`)

Same as the metronome notebook's Section A.  Configure the target datetime/delay,
the search window, and each roamer's **current** route (before the reset).  After
loading the save, read the roamer map + Elm phone to pin the seed.

In [67]:
# --- Section A.1: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)   # <-- your load datetime
a_target_delay   = 681                                     # <-- your load delay
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset).  A roamer roams iff it appears here.
a_prev_routes = {"r": 44, "e": 45, "l": 19}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, display_limit=a_display_limit)
a_seed
# 29 38 19 kpkkpk

Observed roamer routes (R E L, space-separated, . = any):  46 36 1



Observed R=46 E=36 L=1  ->  1 / 183 candidate(s) match

1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02CA  2025-07-24 14:45:55     689    +8   +0   46  36   1   4  PPEEKEKPEEEEPPP

=== Seed identified: 0x0C0E02CA  2025-07-24 14:45:55  delay=689  R/E/L=46/36/1  Elm=PPEEKEKPEEEEPPP ===


{'seed': 202244810,
 'time': datetime.datetime(2025, 7, 24, 14, 45, 55),
 'delay': 689,
 'sec_delta': 0,
 'delay_delta': 8,
 'r_route': 46,
 'e_route': 36,
 'l_route': 1,
 'rng_calls': 4,
 'elm': 'PPEEKEKPEEEEPPP',
 'elm_list': ['P',
  'P',
  'E',
  'E',
  'K',
  'E',
  'K',
  'P',
  'E',
  'E',
  'E',
  'E',
  'P',
  'P',
  'P']}

## Section A.2 — Advance planning  (→ how to reach a Metang)

We are **not** guaranteed a Metang, so we walk Seed A's *advance frame* to one
that yields a Metang (frame 81 = the shiny Metang when we hit the target seed
exactly; otherwise use Pokefinder to pick a Metang frame).

1. **Identify the current advance frame** from the Elm calls you've heard so far
   (1 Elm call = 1 advance).  `max_offset` assumes you paused within ~15 advances
   of the roamer relocation.
2. **Pick the target frame** (Pokefinder handoff — paste the printed Seed A into
   Pokefinder, find a Metang frame, type it back; blank = 81).
3. **Plan the advances**: bulk via chatot flips (2 advances each), then a
   verifiable margin of Elm calls, then Sweet Scent.  The guide shows the Elm
   calls to expect around the target — `]!` marks where to Sweet Scent.

In [68]:
# --- Section A.2: locate the current advance frame, then plan to the target ---
# Regenerate a long Elm sequence for THIS seed (covers the approach to frame ~81+).
a_rng_calls, a_elm = advance_context(a_seed["seed"], a_prev_routes, count=160)

# Type the Elm calls you've heard since the reset (P/E/K); prompts until unique.
a_current_frame = identify_frame(a_rng_calls, a_elm, observed="", max_offset=15)

# Pokefinder handoff for the target encounter frame (blank keeps 81 = shiny Metang).
a_target_frame = prompt_target_frame(a_seed["seed"], default=81)

a_plan  = plan_advances(a_current_frame, a_target_frame)   # margin defaults to 3 Elm calls
a_guide = margin_guide(a_rng_calls, a_elm, a_plan)
print(describe_plan(a_plan, a_guide))

Elm calls: (none)  ->  16 possible frames: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


More Elm calls (type P/E/K as heard):  ee


Elm calls: EE  ->  4 possible frames: [8, 14, 15, 16]


More Elm calls (type P/E/K as heard):  k


Elm calls: EEK  ->  advance frame 9
Seed A: 0x0C0E02CA  -- find a Metang encounter frame in Pokefinder.


Target encounter frame [81]:  28


On advance frame 9; want a Metang encounter on frame 28 (Sweet Scent while on frame 28).
  Advances to go: 19
  1. 8 chatot flips (16 advances) -> land on frame 25
  2. 3 Elm calls -> frame 28, then Sweet Scent.
  Guide: KPEPE[EEE]!EEE   (]! = Sweet Scent here)


## Section B — Safari-compass Seed-B identification  (→ `b_matched`)

Drives the **calibrated** `compass_safari` (frame center from the model, ±kσ over
the RTC-second offsets), exactly as `expedition.compass_safari` does.  Walk the
safari encounter turn by turn — enter `m`/`b`/ball-shakes/`F`/`C` as you see them
— until the candidate set narrows.  Then a confidence check scans for other
nearby seeds that reproduce the same path (aliases), ranked by distance.

The boot key seed and initial time come straight from Section A's identified
`a_seed` (the loaded seed and its datetime) -- no need to re-enter them.  `b_M`
is the commanded countdown = `target_timer_delay + target_timer_calibration`.

In [69]:
# --- Section B: calibrated safari-compass target ---
b_key_seed              = a_seed["seed"]                   # the loaded Seed A (from Section A)
b_initial_time          = a_seed["time"]                   # its datetime (from identify_seed)
b_target_timer_delay    = 479599                           # <-- commanded timer delay (ms)
b_target_timer_calibration = 0                             # <-- timer calibration (ms, signed)
b_max_target_seconds    = 600                              # <-- chart's max target (s)
b_pokemon_name          = "metang"
b_second_offsets        = (-1, 0, 1)   # cover off-by-one timer-start timing (the "3 seconds")
b_confidence_frame_range = 1000          # +/- frames to scan for path-aliases

b_M = b_target_timer_delay + b_target_timer_calibration
model = CalibrationModel.load_default()   # raw metronome fit (data/calibration_model.json)

b_inputs = CompassSafariInput.from_expedition_target(
    # Fold the fitted safari load-path offset into the frame center, exactly as
    # expedition.compass_safari does now -- otherwise Section B centers on the raw metronome
    # frame and drifts ~safari_offset frames off the expedition's target landing.  `model`
    # itself stays RAW so Section D's fit_safari_offset still measures against the metronome
    # fit (folding it there would collapse the offset to ~0 -- a double-correction).
    model=model.with_safari_offset(), M=b_M, initial_time=b_initial_time, key_seed=b_key_seed,
    max_target_seconds=b_max_target_seconds,
    pokemon=safari_pokemon_by_name(b_pokemon_name),
    strategy=STRATEGY_ONLY_BALLS, criteria=CRITERIA_CAPTURE,
    second_offsets=b_second_offsets, mass_cap=0.999,
)

# Interactive: enter the safari path as you play it out.
b_matched = compass_safari(b_inputs)
b_matched

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit                  Metang is angry!
  M / a  Mud, crit (Anger)             Metang is beside itself with anger!
  b      Bait, no crit                 Metang is eating!
  B / e  Bait, crit (Eating)           Metang is busy eating!
  0      Ball, 0 shakes                Oh, no! The Pokémon broke free!
  1      Ball, 1 shake                 Aww! It appeared to be caught!
  2      Ball, 2 shakes                Aargh! Almost had it!
  3      Ball, 3 shakes                Shoot! It was so close, too!
  C      Captured (ends)               Gotcha! Metang was caught!
  F      Fled (ends)                   Metang fled!
  u      Undo last action              —
  ?x     Uncertain result              —
  J      Switch to Jane                —
  w      Widen window & re-apply path  —
  Spaces and commas in input are ignored.


Seeds: 1550 / 1550 remaining
Path:  (none)
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1.


>>  b



Seeds: 1397 / 1550 remaining
Path:  b
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xDE0E7039    28729       0      0     0.36%
   2. 0xDE0E7038    28728      -1      0     0.36%
   3. 0xDE0E703A    28730      +1      0     0.36%
   4. 0xDE0E7037    28727      -2      0     0.36%
   5. 0xDE0E703B    28731      +2      0     0.36%
  Most likely: 0xDE0E7039  P=0.36%  (timer on time)



>>  b



Seeds: 954 / 1550 remaining
Path:  bb
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xDE0E7039    28729       0      0     0.52%
   2. 0xDE0E7038    28728      -1      0     0.52%
   3. 0xDE0E703A    28730      +1      0     0.52%
   4. 0xDE0E703B    28731      +2      0     0.52%
   5. 0xDE0E7036    28726      -3      0     0.52%
  Most likely: 0xDE0E7039  P=0.52%  (timer on time)


...


>>  1



Seeds: 1 / 5915 remaining
Path:  bbbBbb000001mMb101
Balls: 21
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xDE0E6E75    28277    -452      0   100.00%
  Most likely: 0xDE0E6E75  P=100.00%  (timer on time)



>>  F



Pokémon fled. 1 seed(s) matched this path:
Observed path: bbbBbb000001mMb101F
  1. seed=0xDE0E6E75  frame=28277  Δ=-452  δ=0s  P=100.00%


['0xDE0E6E75']

In [70]:
# --- Section B: confidence / neighbor check ---
# The observed path comes straight from compass_safari (b_matched.path) -- no re-entry needed.
b_observed_path = b_matched.path
print(f"Observed path: {b_observed_path}")

if len(b_matched) == 1:
    b_seed = int(b_matched[0], 16)
    b_neighbors = path_confidence(b_inputs, b_seed, b_observed_path,
                                  frame_range=5000)
    print_confidence(b_neighbors)
else:
    b_seed = None
    print(f"{len(b_matched)} seeds still matched -- narrow further before trusting a single seed.")

Observed path: bbbBbb000001mMb101F
No other seed in the scanned window reproduces this path -- high confidence.


## Section C — Save the run  (→ `data/safari_runs.jsonl`)

Appends this run — the identified Seed B (only when a single seed matched), the
Section-A `a_seed` (so the offset fit has `F_a`), the observed path, the commanded
timer (`b_target_timer_delay`, passed straight in), and the calibrated landing
(frame / RTC second / δ) — via the existing `save_safari_run`.  Prompts only for a
**tag** and **notes**, then confirms before writing (every safari run is a fresh
boot, so those fields are fixed).  Saving does **not** touch the calibration model
(that's Section E).

In [71]:
# --- Section C: append this run to data/safari_runs.jsonl ---
# Uses b_target_timer_delay directly (no timer prompt); only prompts for tag, notes, and save.
run_record = save_safari_run(b_matched, inputs=b_inputs, a_seed=a_seed, path=b_observed_path,
                             target_timer_delay=b_target_timer_delay)

Inferred timer offset: on time (δ=0)  (frame 28277, RTC second 485).


Run tag [SCT1]:  SCT2
Notes:  



{
  "saved_at": "2026-09-12T22:39:06",
  "tag": "SCT2",
  "target_timer_delay": 479599,
  "path": "bbbBbb000001mMb101F",
  "n_matched": 1,
  "matched_seeds": [
    "0xDE0E6E75"
  ],
  "seed": 3725487733,
  "seed_hex": "0xDE0E6E75",
  "delay": 28277,
  "frame": 28277,
  "target_frame": 28729,
  "frame_delta": -452,
  "second": 485,
  "second_offset": 0,
  "a_seed": {
    "seed": 202244810,
    "seed_hex": "0x0C0E02CA",
    "time": "2025-07-24T14:45:55",
    "delay": 689,
    "sec_delta": 0,
    "delay_delta": 8,
    "r_route": 46,
    "e_route": 36,
    "l_route": 1,
    "rng_calls": 4,
    "elm": "PPEEKEKPEEEEPPP"
  },
  "notes": ""
}



Save this run? (y/n):  y


Saved to data/safari_runs.jsonl


## Section D — Analysis over `safari_runs.jsonl`

Sparse for now.  Shows the run count and previews the safari **offset** the
current runs imply against the deployed model (does *not* write it).  The
safari-vs-metronome offset measurement proper is tracked in `clayton-abf.10`.

In [72]:
# --- Section D: quick look at the collected safari runs ---
runs = load_safari_runs()
confident = [r for r in runs if r.get("seed") is not None]
print(f"{len(runs)} safari run(s) saved; {len(confident)} with a confident single seed.")

fit = fit_safari_offset(model)   # holds the model slope; median residual = the offset
if fit:
    print(f"Safari offset preview: {fit['offset']:+.2f} frames "
          f"(n={fit['n']}, std={fit['std']:.2f})  -- not written until Section E.")
else:
    print("No usable runs yet (need a_seed + a confident single seed).")

7 safari run(s) saved; 7 with a confident single seed.
Safari offset preview: -463.03 frames (n=7, std=214.22)  -- not written until Section E.


## Section E — Apply the safari offset  (→ `data/calibration_model.json`)

Re-fits the safari **offset only** (holding the metronome slope/β) from
`safari_runs.jsonl`, shows the old → new offset per model, and writes it **only
after you confirm**.  It sets a *separate* `safari_offset` field — the metronome
`alpha`/`beta` are untouched.  The expedition folds this offset into the frame
center for **all** safari scoring (`use_safari_offset`, default True), and so does
Section B above, so after changing it **re-run `precompute_chart()`** (a fast
incremental extend to the shifted frames) **and then `chart_report()`**.

In [49]:
# --- Section E: review the safari offset re-fit, then write it only if confirmed ---
new_models = update_safari_offset()


[linear] safari_offset None -> -437.03 frames  (n=6, std=161.53)
[quad] safari_offset None -> -430.12 frames  (n=6, std=161.53)

*** This shifts only the safari load path (frame(..., safari=True)); the metronome/chart path is unchanged, and no chart rebuild is needed. ***


Write the safari offset? (y/n):  y


Wrote safari offset into ['linear', 'quad'] -> data/calibration_model.json
